In [4]:
import pandas as pd
import numpy as np
import random
import copy
import os
import folium
import time
import requests
import json

# =========================
# Parameter DPSO
# =========================
SWARM_SIZE = 50
MAX_ITER = 1000

# Probabilitas komponen DPSO
P_COG = 0.8       # dorongan menuju pBest
P_SOC = 0.5       # dorongan menuju gBest
P_INERTIA = 0.5   # mempertahankan sebagian velocity sebelumnya

# Early stopping (opsional)
PATIENCE = 80

# Seed supaya hasil bisa direproduksi (opsional)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================
# Parameter Visualisasi
# =========================
klaster_list = ['Barat', 'Pusat', 'Selatan', 'Timur', 'Utara']
warna_klaster = {
    'Barat': 'red',
    'Pusat': 'blue',
    'Selatan': 'green',
    'Timur': 'orange',
    'Utara': 'purple'
}

# Koordinat depot / Gudang Farmasi
depot_coords = [-7.3220, 112.7710]

In [5]:
def calculate_distance(route, dist_matrix):
    """
    Menghitung total jarak:
    depot -> seluruh node dalam route -> kembali ke depot
    Asumsi depot berada pada index 0 di matriks jarak.
    """
    if not route:
        return 0.0

    total_dist = 0.0

    # depot -> node pertama
    total_dist += dist_matrix[0][route[0]]

    # antar node dalam rute
    for i in range(len(route) - 1):
        total_dist += dist_matrix[route[i]][route[i + 1]]

    # node terakhir -> depot
    total_dist += dist_matrix[route[-1]][0]

    return total_dist


def get_swap_sequence(target, current):
    """
    Menghasilkan urutan swap untuk mengubah 'current' menjadi 'target'.
    Ini dipakai sebagai representasi velocity pada DPSO.
    """
    seq = []
    temp = current.copy()

    for i in range(len(target)):
        if temp[i] != target[i]:
            j = temp.index(target[i])
            seq.append((i, j))
            temp[i], temp[j] = temp[j], temp[i]

    return seq


def apply_velocity(route, velocity):
    """
    Menerapkan velocity (list of swap) ke route.
    """
    new_route = route.copy()

    for i, j in velocity:
        new_route[i], new_route[j] = new_route[j], new_route[i]

    return new_route


def run_dpso(file_path,
             swarm_size=SWARM_SIZE,
             max_iter=MAX_ITER,
             p_cog=P_COG,
             p_soc=P_SOC,
             p_inertia=P_INERTIA,
             patience=PATIENCE):
    """
    DPSO untuk optimasi rute:
    - Setiap partikel = permutasi urutan kunjungan node
    - Fitness = total jarak tempuh
    - Velocity = urutan operasi swap
    """

    # -------------------------
    # 1) Load data jarak
    # -------------------------
    df = pd.read_csv(file_path, index_col=0)
    lokasi_list = df.columns.tolist()
    dist_matrix = df.values

    # Asumsi index 0 = depot, sisanya = node/puskesmas
    nodes = list(range(1, len(lokasi_list)))

    if len(nodes) == 0:
        return 0.0, []

    # -------------------------
    # 2) Inisialisasi partikel
    # -------------------------
    swarm = [random.sample(nodes, len(nodes)) for _ in range(swarm_size)]
    velocities = [[] for _ in range(swarm_size)]

    # Inisialisasi pBest
    pbest = copy.deepcopy(swarm)
    pbest_dist = [calculate_distance(route, dist_matrix) for route in swarm]

    # Inisialisasi gBest
    best_idx = int(np.argmin(pbest_dist))
    gbest = copy.deepcopy(pbest[best_idx])
    gbest_dist = pbest_dist[best_idx]

    # Velocity limit (versi discrete dari damping/clamping)
    vmax = int(1.5 * len(nodes))


    # Early stopping
    no_improve = 0

    # -------------------------
    # 3) Iterasi utama DPSO
    # -------------------------
    for _ in range(max_iter):
        improved = False

        # =========================
        # FASE A: Evaluasi fitness
        # =========================
        current_dist = [calculate_distance(route, dist_matrix) for route in swarm]

        # Update pBest
        for i in range(swarm_size):
            if current_dist[i] < pbest_dist[i]:
                pbest[i] = copy.deepcopy(swarm[i])
                pbest_dist[i] = current_dist[i]

        # Update gBest
        best_idx = int(np.argmin(pbest_dist))
        if pbest_dist[best_idx] < gbest_dist:
            gbest = copy.deepcopy(pbest[best_idx])
            gbest_dist = pbest_dist[best_idx]
            improved = True

        # Early stopping
        if improved:
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

        # =========================
        # FASE B: Update velocity & posisi
        # =========================
        for i in range(swarm_size):
            # Komponen cognitive: menuju pBest
            v_cog = get_swap_sequence(pbest[i], swarm[i])

            # Komponen social: menuju gBest
            v_soc = get_swap_sequence(gbest, swarm[i])

            # Gabungkan inertia + cognitive + social
            new_v = (
                [s for s in velocities[i] if random.random() < p_inertia] +
                [s for s in v_cog if random.random() < p_cog] +
                [s for s in v_soc if random.random() < p_soc]
            )

            # Batasi panjang velocity
            if len(new_v) > vmax:
                new_v = random.sample(new_v, vmax)

            velocities[i] = new_v

            # Update posisi partikel
            swarm[i] = apply_velocity(swarm[i], new_v)

    # Konversi index node ke nama lokasi
    route_names = [lokasi_list[node] for node in gbest]

    return gbest_dist, route_names

In [7]:
# Load koordinat puskesmas
df_coords = pd.read_csv("../data/koordinat_puskesmas.csv")

# Rapikan nama kolom dan isi kolom nama
df_coords.columns = df_coords.columns.str.strip()
df_coords['Nama'] = df_coords['Nama'].astype(str).str.strip()

# =========================
# Helper: ambil geometri jalan asli dari OSRM
# =========================
OSRM_BASE_URL = "https://routing.openstreetmap.de/routed-car/route/v1/driving"
# Alternatif:
# OSRM_BASE_URL = "https://router.project-osrm.org/route/v1/driving"

def get_real_road_route(start_latlon, end_latlon, base_url=OSRM_BASE_URL):
    """
    Mengambil geometri rute jalan asli antara dua titik dari OSRM.
    Input  : [lat, lon], [lat, lon]
    Output : (leaflet_coords, distance_km, duration_min)
             leaflet_coords formatnya [ [lat, lon], [lat, lon], ... ]
    """
    start_lat, start_lon = start_latlon
    end_lat, end_lon = end_latlon

    # OSRM memakai urutan lon,lat di URL
    coords = f"{start_lon},{start_lat};{end_lon},{end_lat}"

    params = {
        "overview": "full",
        "geometries": "geojson",
        "steps": "false"
    }

    url = f"{base_url}/{coords}"

    try:
        resp = requests.get(url, params=params, timeout=20)
        resp.raise_for_status()
        data = resp.json()

        if data.get("code") != "Ok" or not data.get("routes"):
            return [], 0.0, 0.0

        route = data["routes"][0]

        # geometry dari OSRM = [lon, lat], dibalik ke [lat, lon] untuk Folium
        geometry_lonlat = route["geometry"]["coordinates"]
        leaflet_coords = [[lat, lon] for lon, lat in geometry_lonlat]

        distance_km = route["distance"] / 1000.0
        duration_min = route["duration"] / 60.0

        return leaflet_coords, distance_km, duration_min

    except Exception as e:
        print(f"   ⚠️ Gagal mengambil rute OSRM: {e}")
        return [], 0.0, 0.0


# Inisialisasi peta
m = folium.Map(
    location=[-7.2575, 112.7521],
    zoom_start=12,
    tiles='CartoDB positron'
)

print("--- HASIL PERHITUNGAN RUTE DPSO ---")

hasil_semua_klaster = []

# Total keseluruhan Surabaya
total_jarak_surabaya = 0
total_runtime_surabaya = 0

# Tambahan total untuk visual jalan asli
total_visual_road_km = 0
total_visual_road_min = 0

for klaster in klaster_list:
    file_csv = f"../data/matriks_jarak_{klaster.lower()}.csv"

    if os.path.exists(file_csv):
        start_time = time.time()
        jarak, rute_nama = run_dpso(file_csv)
        end_time = time.time()

        runtime = end_time - start_time

        # Akumulasi total hasil optimasi DPSO
        total_jarak_surabaya += jarak
        total_runtime_surabaya += runtime

        print(f"✅ {klaster:7}: {jarak:8.2f} km | Runtime: {runtime:.4f} s")
        print(f"   Rute terbaik: Gudang Farmasi -> {' -> '.join(rute_nama)} -> Gudang Farmasi")

        # ------------------------------------------
        # Susun titik rute (depot -> nodes -> depot)
        # ------------------------------------------
        route_points = [depot_coords]

        for nama in rute_nama:
            # Pencocokan nama puskesmas
            match = df_coords[
                df_coords['Nama'].str.replace("Puskesmas ", "", regex=False).str.strip()
                == nama.replace("Puskesmas ", "").strip()
            ]

            if not match.empty:
                pos = [match.iloc[0]['Latitude'], match.iloc[0]['Longitude']]
                route_points.append(pos)

                folium.CircleMarker(
                    location=pos,
                    radius=4,
                    color=warna_klaster[klaster],
                    fill=True,
                    fill_opacity=0.9,
                    tooltip=nama.replace("Puskesmas ", "")
                ).add_to(m)
            else:
                print(f"   ⚠️ Koordinat tidak ditemukan untuk: {nama}")

        # Kembali ke depot
        route_points.append(depot_coords)

        # ------------------------------------------
        # Ambil geometri jalan asli per segmen dari OSRM
        # ------------------------------------------
        full_real_path = []
        real_dist_cluster = 0.0
        real_dur_cluster = 0.0

        for idx in range(len(route_points) - 1):
            start_pt = route_points[idx]
            end_pt = route_points[idx + 1]

            seg_coords, seg_dist_km, seg_dur_min = get_real_road_route(start_pt, end_pt)

            if seg_coords:
                # Hindari duplikasi titik sambungan antarsegmen
                if full_real_path and seg_coords[0] == full_real_path[-1]:
                    full_real_path.extend(seg_coords[1:])
                else:
                    full_real_path.extend(seg_coords)

                real_dist_cluster += seg_dist_km
                real_dur_cluster += seg_dur_min
            else:
                # fallback jika OSRM gagal: tetap gambar garis lurus
                if not full_real_path:
                    full_real_path.append(start_pt)
                full_real_path.append(end_pt)

        total_visual_road_km += real_dist_cluster
        total_visual_road_min += real_dur_cluster

        print(
            f"   ↳ Visual jalan asli {klaster}: "
            f"{real_dist_cluster:.2f} km | {real_dur_cluster:.1f} menit"
        )

        # ------------------------------------------
        # Gambar rute jalan asli di peta
        # ------------------------------------------
        folium.PolyLine(
            full_real_path,
            color=warna_klaster[klaster],
            weight=4,
            opacity=0.8,
            tooltip=(
                f"Rute {klaster} | "
                f"Jarak jalan: {real_dist_cluster:.2f} km | "
                f"Durasi: {real_dur_cluster:.1f} menit"
            )
        ).add_to(m)

        hasil_semua_klaster.append({
            "Klaster": klaster,
            "Jarak Optimasi (km)": jarak,
            "Jarak Visual Jalan (km)": round(real_dist_cluster, 2),
            "Durasi Jalan (menit)": round(real_dur_cluster, 1),
            "Runtime (s)": runtime,
            "Jumlah Titik": len(rute_nama)
        })

    else:
        print(f"⚠️ File tidak ditemukan: {file_csv}")

# Marker depot
folium.Marker(
    depot_coords,
    popup="Gudang Farmasi",
    tooltip="Gudang Farmasi",
    icon=folium.Icon(color='black', icon='home')
).add_to(m)

# Ringkasan hasil per klaster
hasil_df = pd.DataFrame(hasil_semua_klaster)

# Tambahkan baris total
baris_total = pd.DataFrame([{
    "Klaster": "TOTAL SURABAYA",
    "Jarak Optimasi (km)": round(total_jarak_surabaya, 2),
    "Jarak Visual Jalan (km)": round(total_visual_road_km, 2),
    "Durasi Jalan (menit)": round(total_visual_road_min, 1),
    "Runtime (s)": round(total_runtime_surabaya, 4),
    "Jumlah Titik": hasil_df["Jumlah Titik"].sum() if not hasil_df.empty else 0
}])

hasil_df_final = pd.concat([hasil_df, baris_total], ignore_index=True)

display(hasil_df_final)

print("\n=== TOTAL KESELURUHAN SURABAYA ===")
print(f"Total jarak optimasi DPSO : {total_jarak_surabaya:.2f} km")
print(f"Total jarak visual jalan  : {total_visual_road_km:.2f} km")
print(f"Total durasi visual jalan : {total_visual_road_min:.1f} menit")
print(f"Total runtime seluruh klaster: {total_runtime_surabaya:.4f} s")

# Menyiapkan data untuk disimpan
output_data = {
    "total_summary": {
        "total_distance_km": round(total_jarak_surabaya, 2),
        "total_duration_min": round(total_visual_road_min, 1),
        "total_runtime_s": round(total_runtime_surabaya, 4)
    },
    "cluster_details": hasil_semua_klaster
}

# Membuat folder output jika belum ada
os.makedirs("../output_json", exist_ok=True)

# Simpan ke file
with open("../output_json/rute_dpso.json", "w") as f:
    json.dump(output_data, f, indent=4)

print("✅ Berhasil menyimpan hasil ke output_json/rute_dpso.json")

# Tampilkan peta
m

--- HASIL PERHITUNGAN RUTE DPSO ---
✅ Barat  :    44.98 km | Runtime: 0.0446 s
   Rute terbaik: Gudang Farmasi -> Puskesmas Simomulyo Surabaya -> Puskesmas Asemrowo Surabaya -> Puskesmas Tanjungsari Surabaya -> Puskesmas Balongsari Surabaya -> Puskesmas Manukan Kulon Surabaya -> Puskesmas Sememi Surabaya -> Puskesmas Benowo Surabaya -> Puskesmas Made Surabaya -> Puskesmas Lontar Surabaya -> Puskesmas Lidah Kulon Surabaya -> Puskesmas Jeruk Surabaya -> Puskesmas Bangkingan Surabaya -> Gudang Farmasi
   ↳ Visual jalan asli Barat: 73.71 km | 102.5 menit
✅ Pusat  :    26.11 km | Runtime: 0.0290 s
   Rute terbaik: Gudang Farmasi -> Puskesmas Ketabang Surabaya -> Puskesmas Tambakrejo Surabaya -> Puskesmas Simolawang Surabaya -> Puskesmas Peneleh Surabaya -> Puskesmas Gundih Surabaya -> Puskesmas Tembok Dukuh Surabaya -> Puskesmas Kedungdoro Surabaya -> Puskesmas Dr. Soetomo Surabaya -> Gudang Farmasi
   ↳ Visual jalan asli Pusat: 44.56 km | 62.9 menit
✅ Selatan:    36.65 km | Runtime: 0.0755

,Klaster,Jarak Optimasi (km),Jarak Visual Jalan (km),Durasi Jalan (menit),Runtime (s),Jumlah Titik
0,Barat,44.983,73.71,102.5,0.044580,12
1,Pusat,26.111,44.56,62.9,0.029020,8
2,Selatan,36.652,75.52,103.8,0.075497,16
3,Timur,33.975,60.72,88.7,0.085904,14
4,Utara,34.851,60.17,80.1,0.096017,13
5,TOTAL SURABAYA,176.570,314.68,438.0,0.331000,63



=== TOTAL KESELURUHAN SURABAYA ===
Total jarak optimasi DPSO : 176.57 km
Total jarak visual jalan  : 314.68 km
Total durasi visual jalan : 438.0 menit
Total runtime seluruh klaster: 0.3310 s
✅ Berhasil menyimpan hasil ke output_json/rute_dpso.json
